# Practical Guide to Pydantic

In [1]:
from pydantic import BaseModel

class Student(BaseModel):
    name: str
    age: int
    course: str

In [2]:
student = Student(
    name="Sara",
    age=21,
    course="Data Science"
)

print(student)

name='Sara' age=21 course='Data Science'


In [3]:
print(student.name)
print(student.age)
print(type(student.age))

Sara
21
<class 'int'>


In [4]:
from pydantic import ValidationError

try:
    student = Student(
        name="Sara",
        age="twenty-one",
        course="Data Science"
    )

except ValidationError as e:
    print("Validation failed!")
    print(e)

Validation failed!
1 validation error for Student
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='twenty-one', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


In [5]:
student = Student(
    name="Sara",
    age="21",
    course="Data Science"
)

print(student.age)
print(type(student.age))

21
<class 'int'>


In [6]:
from pydantic import BaseModel, Field

class Book(BaseModel):
    title: str
    author: str
    publication_year: int = Field(ge=1900, le=2026)
    available_copies: int = Field(default=0, ge=0)

In [7]:
book = Book(
    title="Python for Beginners",
    author="Alex Brown",
    publication_year=2024,
    available_copies=5
)

print(book)

title='Python for Beginners' author='Alex Brown' publication_year=2024 available_copies=5


In [8]:
book = Book(
    title="Learning AI",
    author="Maya Lee",
    publication_year=2025
)

print(book.available_copies)

0


In [9]:
try:
    book = Book(
        title="Learning AI",
        author="Maya Lee",
        publication_year=1800,
        available_copies=-3
    )

except ValidationError as e:
    print("Book validation failed!")

    for error in e.errors():
        print(error)

Book validation failed!
{'type': 'greater_than_equal', 'loc': ('publication_year',), 'msg': 'Input should be greater than or equal to 1900', 'input': 1800, 'ctx': {'ge': 1900}, 'url': 'https://errors.pydantic.dev/2.13/v/greater_than_equal'}
{'type': 'greater_than_equal', 'loc': ('available_copies',), 'msg': 'Input should be greater than or equal to 0', 'input': -3, 'ctx': {'ge': 0}, 'url': 'https://errors.pydantic.dev/2.13/v/greater_than_equal'}


In [10]:
from pydantic import BaseModel

class Lesson(BaseModel):
    title: str
    duration_minutes: int

In [11]:
lesson = Lesson(
    title="Introduction to Python",
    duration_minutes=30
)

print(lesson)

title='Introduction to Python' duration_minutes=30


In [12]:
class Course(BaseModel):
    name: str
    instructor: str
    lessons: list[Lesson]

In [13]:
course_data = {
    "name": "Python Basics",
    "instructor": "Maya Lee",
    "lessons": [
        {
            "title": "Variables and Data Types",
            "duration_minutes": 20
        },
        {
            "title": "Functions",
            "duration_minutes": 35
        }
    ]
}

In [14]:
course = Course.model_validate(course_data)

print(course)

name='Python Basics' instructor='Maya Lee' lessons=[Lesson(title='Variables and Data Types', duration_minutes=20), Lesson(title='Functions', duration_minutes=35)]


In [15]:
print(course.name)

print(course.lessons[0].title)

print(type(course.lessons[0]))

Python Basics
Variables and Data Types
<class '__main__.Lesson'>


In [16]:
course_dict = course.model_dump()

print(course_dict)
print(type(course_dict))

{'name': 'Python Basics', 'instructor': 'Maya Lee', 'lessons': [{'title': 'Variables and Data Types', 'duration_minutes': 20}, {'title': 'Functions', 'duration_minutes': 35}]}
<class 'dict'>


In [17]:
from typing import Literal
from pydantic import BaseModel, Field

class ClassificationOutput(BaseModel):
    category: Literal["delivery", "billing", "technical", "other"]
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float = Field(ge=0, le=1)

In [18]:
category: Literal["delivery", "billing", "technical", "other"]

In [19]:
confidence: float = Field(ge=0, le=1)

In [20]:
model_output = {
    "category": "delivery",
    "sentiment": "negative",
    "confidence": 0.95
}

In [21]:
result = ClassificationOutput.model_validate(model_output)

print(result)

category='delivery' sentiment='negative' confidence=0.95


In [22]:
print("Category:", result.category)
print("Sentiment:", result.sentiment)
print("Confidence:", result.confidence)

Category: delivery
Sentiment: negative
Confidence: 0.95


In [23]:
bad_model_output = {
    "category": "shipping",
    "sentiment": "negative",
    "confidence": 1.5
}

In [24]:
try:
    result = ClassificationOutput.model_validate(bad_model_output)

except ValidationError as e:
    print("The model output is invalid!")

    for error in e.errors():
        print(error)

The model output is invalid!
{'type': 'literal_error', 'loc': ('category',), 'msg': "Input should be 'delivery', 'billing', 'technical' or 'other'", 'input': 'shipping', 'ctx': {'expected': "'delivery', 'billing', 'technical' or 'other'"}, 'url': 'https://errors.pydantic.dev/2.13/v/literal_error'}
{'type': 'less_than_equal', 'loc': ('confidence',), 'msg': 'Input should be less than or equal to 1', 'input': 1.5, 'ctx': {'le': 1.0}, 'url': 'https://errors.pydantic.dev/2.13/v/less_than_equal'}


In [25]:
model_response = """
{
    "category": "billing",
    "sentiment": "neutral",
    "confidence": 0.88
}
"""

result = ClassificationOutput.model_validate_json(model_response)

print(result)

category='billing' sentiment='neutral' confidence=0.88


In [26]:
# Python dictionary
ClassificationOutput.model_validate({
    "category": "billing",
    "sentiment": "neutral",
    "confidence": 0.88
})

ClassificationOutput(category='billing', sentiment='neutral', confidence=0.88)

In [27]:
# JSON string
ClassificationOutput.model_validate_json("""
{
    "category": "billing",
    "sentiment": "neutral",
    "confidence": 0.88
}
""")

ClassificationOutput(category='billing', sentiment='neutral', confidence=0.88)

In [28]:
print(result.model_dump())

{'category': 'billing', 'sentiment': 'neutral', 'confidence': 0.88}


In [29]:
print(result.model_dump_json(indent=2))

{
  "category": "billing",
  "sentiment": "neutral",
  "confidence": 0.88
}


# Exercise 1: Create a Movie Model


In [30]:
# Exercise 1: Create a Movie Model

from pydantic import BaseModel, Field, ValidationError

class Movie(BaseModel):
    title: str
    director: str
    rating: float = Field(ge=0, le=10)
    release_year: int = Field(ge=1900, le=2026)

In [31]:
# Valid movie
movie = Movie(
    title="Interstellar",
    director="Christopher Nolan",
    rating=8.7,
    release_year=2014
)

print(movie)


title='Interstellar' director='Christopher Nolan' rating=8.7 release_year=2014


In [32]:
# Invalid rating
try:
    movie = Movie(
        title="Interstellar",
        director="Christopher Nolan",
        rating=12,
        release_year=2014
    )

except ValidationError as e:
    for error in e.errors():
        print(error)

{'type': 'less_than_equal', 'loc': ('rating',), 'msg': 'Input should be less than or equal to 10', 'input': 12, 'ctx': {'le': 10.0}, 'url': 'https://errors.pydantic.dev/2.13/v/less_than_equal'}


In [33]:
# Missing title
try:
    movie = Movie(
        director="Christopher Nolan",
        rating=8.7,
        release_year=2014
    )

except ValidationError as e:
    for error in e.errors():
        print(error)

{'type': 'missing', 'loc': ('title',), 'msg': 'Field required', 'input': {'director': 'Christopher Nolan', 'rating': 8.7, 'release_year': 2014}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}


# Exercise2: Create a Nested Model


In [34]:
from pydantic import BaseModel

class Ingredient(BaseModel):
    name: str
    quantity: float

class Recipe(BaseModel):
    name: str
    ingredients: list[Ingredient]

In [35]:
#Create a recipe
recipe = Recipe(
    name="Pancakes",
    ingredients=[
        Ingredient(name="Flour", quantity=200),
        Ingredient(name="Milk", quantity=300),
        Ingredient(name="Eggs", quantity=2)
    ]
)

print(recipe)

name='Pancakes' ingredients=[Ingredient(name='Flour', quantity=200.0), Ingredient(name='Milk', quantity=300.0), Ingredient(name='Eggs', quantity=2.0)]


In [36]:
# Export it to JSON
recipe_json = recipe.model_dump_json(indent=2)

print(recipe_json)

{
  "name": "Pancakes",
  "ingredients": [
    {
      "name": "Flour",
      "quantity": 200.0
    },
    {
      "name": "Milk",
      "quantity": 300.0
    },
    {
      "name": "Eggs",
      "quantity": 2.0
    }
  ]
}


# Exercise 3: Validate a fine-tuned model output


In [37]:
from pydantic import BaseModel, Field
from typing import Literal

class SentimentOutput(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float = Field(ge=0, le=1)
    explanation: str

In [38]:
valid_output = {
    "sentiment": "positive",
    "confidence": 0.92,
    "explanation": "The customer is happy with the service."
}

result = SentimentOutput(**valid_output)

print(result)

sentiment='positive' confidence=0.92 explanation='The customer is happy with the service.'


In [39]:
from pydantic import ValidationError

invalid_output = {
    "sentiment": "very positive",
    "confidence": 1.4,
    "explanation": "The customer is happy."
}

try:
    result = SentimentOutput(**invalid_output)
    print(result)
except ValidationError as e:
    print(e)

2 validation errors for SentimentOutput
sentiment
  Input should be 'positive', 'negative' or 'neutral' [type=literal_error, input_value='very positive', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.4, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


### Why the invalid output fails

The invalid output fails validation for two reasons:

1. The value `"very positive"` is not allowed for `sentiment`. The model only accepts `"positive"`, `"negative"`, or `"neutral"`.

2. The `confidence` value is `1.4`, but confidence must be between `0` and `1`.

Pydantic detects both invalid values and returns validation errors instead of accepting the data.